In [140]:
!pip install optuna

In [141]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [142]:
df = pd.read_csv('/content/diabetes.csv')

In [143]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [144]:
import numpy as np

cols_with_missing_vals=['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols_with_missing_vals]=df[cols_with_missing_vals].replace(0,np.nan)

df.fillna(df.median(),inplace=True)
df.isnull().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [145]:
X=df.drop('Outcome',axis=1)
y=df['Outcome']

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [146]:
X_train.shape

(537, 8)

In [147]:
X_test.shape

(231, 8)

In [148]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [149]:
def objective(trial):
  n_estimators=trial.suggest_int('n_estimators',50,200)
  max_depth=trial.suggest_int('max_depth',1,20)

  model=RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )

  score=cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()
  return score

In [160]:
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-08-11 10:27:05,037] A new study created in memory with name: no-name-24beceac-5149-4271-aab6-344b82a6ec73
[I 2026-08-11 10:27:05,609] Trial 0 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-08-11 10:27:06,521] Trial 1 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-08-11 10:27:06,857] Trial 2 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-08-11 10:27:07,462] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 0 with value: 0.7653631284916201.
[I 2026-08-11 10:27:08,105] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 0 with value: 0.765363

In [161]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler())
study.optimize(objective,n_trials=50)

[I 2026-08-11 10:27:21,123] A new study created in memory with name: no-name-a45b87b8-497b-4b48-a72b-4847cbde8a74
[I 2026-08-11 10:27:22,060] Trial 0 finished with value: 0.7597765363128492 and parameters: {'n_estimators': 145, 'max_depth': 8}. Best is trial 0 with value: 0.7597765363128492.
[I 2026-08-11 10:27:22,958] Trial 1 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 138, 'max_depth': 17}. Best is trial 1 with value: 0.7616387337057727.
[I 2026-08-11 10:27:23,706] Trial 2 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 115, 'max_depth': 14}. Best is trial 2 with value: 0.7672253258845437.
[I 2026-08-11 10:27:24,421] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 118, 'max_depth': 19}. Best is trial 2 with value: 0.7672253258845437.
[I 2026-08-11 10:27:25,696] Trial 4 finished with value: 0.7579143389199255 and parameters: {'n_estimators': 153, 'max_depth': 6}. Best is trial 2 with value: 0.767225

In [162]:
print(f'best trial accuracy : {study.best_value}')
print(f'best hyperparameters : {study.best_params}')

best trial accuracy : 0.7783985102420856
best hyperparameters : {'n_estimators': 109, 'max_depth': 7}


In [163]:
from sklearn.metrics import accuracy_score

best_model=RandomForestClassifier(**study.best_params,random_state=42)

best_model.fit(X_train,y_train)

y_pred = best_model.predict(X_test)

test_accuracy=accuracy_score(y_test,y_pred)

print(f'Test Accuracy with best Parameter is :{test_accuracy}')

Test Accuracy with best Parameter is :0.7489177489177489


In [164]:
search_space = {
    'n_estimators':[50,100,150,200],
    'max_depth':[5,10,15,20]
}

In [165]:
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_contour,plot_param_importances

In [166]:
plot_optimization_history(study)

In [167]:
plot_parallel_coordinate(study)

In [168]:
plot_slice(study)

In [169]:
plot_contour(study)

In [170]:
plot_param_importances(study)

In [171]:
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC

In [172]:
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [173]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=100)

[I 2026-08-11 10:38:49,062] A new study created in memory with name: no-name-a527b10f-b221-4974-9c0c-043a08255457
[I 2026-08-11 10:38:49,141] Trial 0 finished with value: 0.7635009310986964 and parameters: {'classifier': 'SVM', 'C': 0.2603711137997444, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-08-11 10:38:50,462] Trial 1 finished with value: 0.7765363128491621 and parameters: {'classifier': 'RandomForest', 'n_estimators': 166, 'max_depth': 19, 'min_samples_split': 9, 'min_samples_leaf': 7, 'bootstrap': False}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-08-11 10:38:51,707] Trial 2 finished with value: 0.7653631284916201 and parameters: {'classifier': 'RandomForest', 'n_estimators': 91, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 1 with value: 0.7765363128491621.
[I 2026-08-11 10:38:51,761] Trial 3 finished with value: 0.7001862197392924 and parameters: {'classifier': '

In [174]:
study.best_params

{'classifier': 'SVM',
 'C': 0.10720085704542717,
 'kernel': 'linear',
 'gamma': 'scale'}

In [175]:
study.best_value

0.7858472998137803

In [176]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.763501,2026-08-11 10:38:49.070367,2026-08-11 10:38:49.141606,0 days 00:00:00.071239,0.260371,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.776536,2026-08-11 10:38:49.143414,2026-08-11 10:38:50.462715,0 days 00:00:01.319301,NaN,False,RandomForest,NaN,NaN,NaN,19.0,7.0,9.0,166.0,COMPLETE
2,2,0.765363,2026-08-11 10:38:50.466395,2026-08-11 10:38:51.707772,0 days 00:00:01.241377,NaN,True,RandomForest,NaN,NaN,NaN,16.0,8.0,2.0,91.0,COMPLETE
3,3,0.700186,2026-08-11 10:38:51.710075,2026-08-11 10:38:51.761781,0 days 00:00:00.051706,5.119073,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.776536,2026-08-11 10:38:51.764491,2026-08-11 10:38:53.271228,0 days 00:00:01.506737,NaN,NaN,GradientBoosting,NaN,NaN,0.022633,17.0,8.0,10.0,64.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.765363,2026-08-11 10:39:34.437809,2026-08-11 10:39:36.332401,0 days 00:00:01.894592,NaN,NaN,GradientBoosting,NaN,NaN,0.028379,7.0,7.0,4.0,136.0,COMPLETE
96,96,0.783985,2026-08-11 10:39:36.333797,2026-08-11 10:39:36.384239,0 days 00:00:00.050442,0.210405,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.752328,2026-08-11 10:39:36.385432,2026-08-11 10:39:37.102104,0 days 00:00:00.716672,NaN,True,RandomForest,NaN,NaN,NaN,3.0,2.0,9.0,80.0,COMPLETE
98,98,0.754190,2026-08-11 10:39:37.103366,2026-08-11 10:39:37.168450,0 days 00:00:00.065084,0.157709,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [180]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,74
RandomForest,15
GradientBoosting,11


In [181]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.751143
RandomForest,0.767970
SVM,0.773567


In [182]:
plot_optimization_history(study).show()


In [183]:
plot_slice(study).show()